In [1]:
# Importación de librerías especializadas para ingeniería de características en transformadores
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
from datetime import datetime, timedelta
import gc
import sys

# Librerías para análisis avanzado de series temporales
from scipy import signal, stats
from scipy.fft import fft, fftfreq
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.decomposition import PCA
from sklearn.feature_selection import mutual_info_regression, SelectKBest, f_regression

# Librerías para visualización técnica
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.dates import DateFormatter
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Configuración del entorno para análisis de transformadores
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', '{:.4f}'.format)
plt.style.use('default')
sns.set_palette("husl")

# Configuración para reproducibilidad
np.random.seed(42)

print(" INGENIERÍA DE CARACTERÍSTICAS PARA TRANSFORMADORES ELÉCTRICOS")
print("=" * 65)
print(f" Pandas: {pd.__version__}")
print(f" NumPy: {np.__version__}")
print(f" SciPy: {signal.__version__ if hasattr(signal, '__version__') else 'importado'}")
print(f" Configuración optimizada para análisis de transformadores de potencia")
print(f" Objetivo: Predicción de fallas con horizonte de 30 días")
print(f" Fecha de procesamiento: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

 INGENIERÍA DE CARACTERÍSTICAS PARA TRANSFORMADORES ELÉCTRICOS
 Pandas: 2.2.3
 NumPy: 2.2.6
 SciPy: importado
 Configuración optimizada para análisis de transformadores de potencia
 Objetivo: Predicción de fallas con horizonte de 30 días
 Fecha de procesamiento: 2025-09-26 05:03:56


In [4]:
# ================================
# Section 1 — Paths & Parameters
# ================================
from pathlib import Path
from datetime import datetime

from pathlib import Path

def find_repo_root() -> Path:
    p = Path.cwd().resolve()
    for cand in (p, *p.parents):
        if (cand / "data").exists():
            return cand
    return p

# Paths SILVER (input) and GOLD (output)
BASE_DIR = find_repo_root()
RUTA_SILVER = BASE_DIR / "data" / "capa_silver" / "preprocesamiento_silver"
RUTA_PROCESSED = BASE_DIR / "data" / "processed"

# --- Gold paths (Delta outputs for features) ---
RUTA_GOLD_BASE = BASE_DIR / "data" / "capa_gold" / "features_transformador"
RUTA_GOLD_COMPLETE = RUTA_GOLD_BASE / "features_complete"
RUTA_GOLD_TRAIN    = RUTA_GOLD_BASE / "features_train"
RUTA_GOLD_VALID    = RUTA_GOLD_BASE / "features_valid"
RUTA_GOLD_TEST     = RUTA_GOLD_BASE / "features_test"

# --- Reports for feature engineering (human-readable artifacts) ---
RUTA_REPORTS_FE = BASE_DIR / "reports" / "feature_engineering"

# Create directories if they don't exist
for ruta in [RUTA_GOLD_BASE, RUTA_GOLD_COMPLETE, RUTA_GOLD_TRAIN, RUTA_GOLD_VALID, RUTA_GOLD_TEST, RUTA_REPORTS_FE]:
    ruta.mkdir(parents=True, exist_ok=True)

# --- Technical parameters (aligned with your config + domain defaults) ---
# Use your known specs where possible; keep FE horizons/windows explicit here.
PARAMETROS_TRANSFORMADOR = {
    # Prediction horizon & analysis window
    "horizonte_prediccion_dias": 30,   # target horizon for preventive maintenance labels
    "ventana_analisis_dias": 90,       # lookback window for temporal features
    "frecuencia_muestreo": "H",        # hourly frequency (Silver is hourly)

    # Rolling windows (hours)
    "ventanas_rolling": [24, 72, 168, 720],  # 1d, 3d, 7d, 30d

    # Lags (hours)
    "lags_corto_plazo":  [1, 6, 12, 24],     # immediate/inertia effects
    "lags_mediano_plazo": [48, 72, 168],     # operational cycles
    "lags_largo_plazo":  [336, 720],         # degradation trends

    # Nominal specs (prefer project config; fallback to sensible defaults)
    "capacidad_nominal_mva": 100,     # from config_silverv2 SPECS 
    "voltaje_nominal_kv": 138,        # primary side nominal 
    "temp_aceite_nominal": 65,       # typical nominal oil temp
    "temp_ambiente_nominal": 40,     # design ambient (from config_silverv2)

    # Thresholds for composite features/anomaly-inspired indicators
    "umbral_gradiente_termico": 25,  # hotspot - oil [°C]
    "umbral_sobrecarga": 1.2,        # load factor p.u.
    "umbral_tap_excesivo": 5,        # tap ops per day

    # Frequency-domain options (optional; keep for future FFT feature block)
    "freq_nyquist": 0.5,             # Nyquist for hourly sampling
    "n_componentes_fft": 10,         # top FFT components to extract
}

# --- Pretty print configuration for the notebook run ---
print(" CONFIGURACIÓN DE PARÁMETROS TÉCNICOS")
print("-" * 45)
print(f" 1. Horizonte de predicción: {PARAMETROS_TRANSFORMADOR['horizonte_prediccion_dias']} días")
print(f" 2. Ventana de análisis:     {PARAMETROS_TRANSFORMADOR['ventana_analisis_dias']} días")
print(f" 3. Capacidad nominal:       {PARAMETROS_TRANSFORMADOR['capacidad_nominal_mva']} MVA")
print(f" 4. Umbral gradiente térmico: {PARAMETROS_TRANSFORMADOR['umbral_gradiente_termico']} °C")
print(f" 5. Ventanas rolling (h):    {PARAMETROS_TRANSFORMADOR['ventanas_rolling']}")

print("\nRUTAS DE TRABAJO")
print(f"   Silver Delta: {RUTA_SILVER} - {' Existe' if RUTA_SILVER.exists() else '❌ No existe'}")
print(f"    Processed:    {RUTA_PROCESSED} - {' Existe' if RUTA_PROCESSED.exists() else '❌ No existe'}")
print(f"    Gold (base):  {RUTA_GOLD_BASE} - {' Creada' if RUTA_GOLD_BASE.exists() else '❌ Error'}")
print(f"    Reports FE:   {RUTA_REPORTS_FE} - {' Creada' if RUTA_REPORTS_FE.exists() else '❌ Error'}")


 CONFIGURACIÓN DE PARÁMETROS TÉCNICOS
---------------------------------------------
 1. Horizonte de predicción: 30 días
 2. Ventana de análisis:     90 días
 3. Capacidad nominal:       100 MVA
 4. Umbral gradiente térmico: 25 °C
 5. Ventanas rolling (h):    [24, 72, 168, 720]

RUTAS DE TRABAJO
   Silver Delta: C:\Users\Asus TUF\Desktop\proy_ml\data\capa_silver\preprocesamiento_silver -  Existe
    Processed:    C:\Users\Asus TUF\Desktop\proy_ml\data\processed -  Existe
    Gold (base):  C:\Users\Asus TUF\Desktop\proy_ml\data\capa_gold\features_transformador -  Creada
    Reports FE:   C:\Users\Asus TUF\Desktop\proy_ml\reports\feature_engineering -  Creada


In [5]:
# ================================
# Section 2 — Load preprocessed data (Silver → FE inputs)
# ================================
from pathlib import Path
import pandas as pd
from deltalake import DeltaTable

# Usamos las rutas ya definidas en la sección anterior
# RUTA_SILVER, RUTA_PROCESSED vienen de config_silverv2

def cargar_datos_transformador_preprocesados(ruta_silver: Path, ruta_processed: Path) -> dict:
    """
    Load and validate preprocessed transformer datasets.
    Priority: read from Silver Delta table. Fallback: read processed parquet files (if present).
    
    Returns:
        dict with:
          - df_principal: pd.DataFrame (indexed by timestamp, hourly)
          - df_anomalias: pd.DataFrame (subset ALERTA/CRITICO; derived if not saved separately)
          - metadatos: str | None
          - info_variables: pd.DataFrame | None
    """
    print(" CARGA DE DATOS PREPROCESADOS DE TRANSFORMADORES (Delta First)")
    print("=" * 70)

    # Archivos esperados si hay fallback a 'processed'
    archivos_esperados = {
        "principal": "transformador_data.parquet",
        "anomalias": "transformador_data_anomalias.parquet",   # puede no existir en Silver v2 modular
        "metadatos": "transformador_data_metadatos_tecnicos.txt",
        "variables": "transformador_data_variables.csv",
    }

    # -----------------------------
    # 1) Intentar lectura desde Silver (Delta)
    # -----------------------------
    df_principal = None
    if ruta_silver.exists():
        try:
            print(f"   Leyendo Silver (Delta): {ruta_silver}")
            dt = DeltaTable(str(ruta_silver))
            df_principal = dt.to_pandas()
        except Exception as e:
            print(f"   No se pudo leer Delta Silver: {e}")

    # -----------------------------
    # 2) Fallback: processed/parquet
    # -----------------------------
    if df_principal is None:
        print("    Fallback a archivos en processed/")
        # principal
        ruta_pq = ruta_processed / archivos_esperados["principal"]
        if not ruta_pq.exists():
            raise FileNotFoundError(
                "No se encontró Silver Delta ni el parquet principal en processed/"
            )
        df_principal = pd.read_parquet(ruta_pq)

    # -----------------------------
    # 3) Normalizar índice temporal
    # -----------------------------
    if "timestamp" in df_principal.columns:
        df_principal["timestamp"] = pd.to_datetime(df_principal["timestamp"], utc=True, errors="coerce")
        df_principal = df_principal.sort_values("timestamp").set_index("timestamp")
    elif not isinstance(df_principal.index, pd.DatetimeIndex):
        df_principal.index = pd.to_datetime(df_principal.index, utc=True, errors="coerce")
        df_principal = df_principal.sort_index()

    # Resumen
    print(f"   Dimensiones: {len(df_principal):,} × {df_principal.shape[1]}")
    if len(df_principal):
        print(f"   Período: {df_principal.index.min()} → {df_principal.index.max()}")
        print(f"   ⏱Duración: {df_principal.index.max() - df_principal.index.min()}")

    # -----------------------------
    # 4) Dataset de anomalías
    #    - Preferir parquet si Existe (notebook 02)
    #    - Si no, derivar de df_principal (Silver v2 modular)
    # -----------------------------
    df_anomalias = None
    ruta_anom = ruta_processed / archivos_esperados["anomalias"]
    if ruta_anom.exists():
        try:
            print("    Cargando dataset de anomalías (processed/parquet)...")
            df_anomalias = pd.read_parquet(ruta_anom)
            if "timestamp" in df_anomalias.columns:
                df_anomalias["timestamp"] = pd.to_datetime(df_anomalias["timestamp"], utc=True, errors="coerce")
                df_anomalias = df_anomalias.sort_values("timestamp").set_index("timestamp")
            elif not isinstance(df_anomalias.index, pd.DatetimeIndex):
                df_anomalias.index = pd.to_datetime(df_anomalias.index, utc=True, errors="coerce")
                df_anomalias = df_anomalias.sort_index()
        except Exception as e:
            print(f"    Error leyendo anomalías parquet: {e}")

    # Derivar si no hay parquet
    if df_anomalias is None and "estado_operacional" in df_principal.columns:
        print("   Derivando anomalías desde df_principal (ALERTA/CRITICO)...")
        df_anomalias = df_principal[df_principal["estado_operacional"].isin(["ALERTA", "CRITICO"])].copy()

    # -----------------------------
    # 5) Distribución de estados (si existe)
    # -----------------------------
    if "estado_operacional" in df_principal.columns:
        estados = df_principal["estado_operacional"].value_counts(dropna=False)
        print("\nDistribución de estados operacionales:")
        for estado, cantidad in estados.items():
            porcentaje = (cantidad / len(df_principal) * 100) if len(df_principal) else 0.0
            emoji = {"NORMAL": "✅", "ALERTA": "⚠️", "CRITICO": "🚨"}.get(str(estado), "📊")
            print(f"   {emoji} {estado}: {cantidad:,} ({porcentaje:.1f}%)")

    # -----------------------------
    # 6) Metadatos técnicos (TXT) — opcional
    # -----------------------------
    metadatos = None
    ruta_meta = ruta_processed / archivos_esperados["metadatos"]
    if ruta_meta.exists():
        try:
            metadatos = ruta_meta.read_text(encoding="utf-8")
            print("\nMetadatos técnicos cargados")
        except Exception as e:
            print(f"   Error cargando metadatos: {e}")

    # -----------------------------
    # 7) Info de variables (CSV) — opcional
    # -----------------------------
    info_variables = None
    ruta_vars = ruta_processed / archivos_esperados["variables"]
    if ruta_vars.exists():
        try:
            info_variables = pd.read_csv(ruta_vars)
            print("\n Información de variables:")
            cats = info_variables.groupby("categoria_tecnica")["variable"].count()
            for categoria, qty in cats.items():
                print(f"   • {categoria}: {qty} variables")
        except Exception as e:
            print(f"   Error cargando info de variables: {e}")

    print("\n CARGA COMPLETADA — datos listos para Feature Engineering")
    return {
        "df_principal": df_principal,
        "df_anomalias": df_anomalias,
        "metadatos": metadatos,
        "info_variables": info_variables,
    }

# Ejecutar carga (usando Delta primero, luego fallback a processed si hace falta)
datos_cargados = cargar_datos_transformador_preprocesados(RUTA_SILVER, RUTA_PROCESSED)
df_transformador = datos_cargados["df_principal"]
df_anomalias = datos_cargados["df_anomalias"]
metadatos_tecnicos = datos_cargados["metadatos"]
info_variables = datos_cargados["info_variables"]


 CARGA DE DATOS PREPROCESADOS DE TRANSFORMADORES (Delta First)
   Leyendo Silver (Delta): C:\Users\Asus TUF\Desktop\proy_ml\data\capa_silver\preprocesamiento_silver
   Dimensiones: 8,442 × 15
   Período: 2024-09-10 04:00:00+00:00 → 2025-08-27 21:00:00+00:00
   ⏱Duración: 351 days 17:00:00
   Derivando anomalías desde df_principal (ALERTA/CRITICO)...

Distribución de estados operacionales:
   🚨 CRITICO: 8,429 (99.8%)
   ✅ NORMAL: 13 (0.2%)

Metadatos técnicos cargados

 Información de variables:
   • Clasificación: 3 variables
   • Eléctrica: 3 variables
   • Mecánica: 1 variables
   • Otra: 1 variables
   • Térmica: 5 variables

 CARGA COMPLETADA — datos listos para Feature Engineering


In [7]:
# ================================
# Section 2 — Load preprocessed data (Delta First) + Pretty Summary
# ================================
from pathlib import Path
import pandas as pd
from deltalake import DeltaTable

ESTADO_EMOJI = {"NORMAL": "✅", "ALERTA": "⚠️", "CRITICO": "🚨"}

def cargar_datos_transformador_preprocesados(ruta_silver: Path, ruta_processed: Path) -> dict:
    """
    Load and validate preprocessed transformer datasets.
    Priority: read from Silver Delta table. Fallback: read processed parquet files (if present).
    Prints a friendly summary block with states distribution, metadata & variables info.
    """
    print("⚡ CARGA DE DATOS PREPROCESADOS DE TRANSFORMADORES (Delta First)")
    print("=" * 70)

    archivos_esperados = {
        "principal": "transformador_data.parquet",
        "anomalias": "transformador_data_anomalias.parquet",
        "metadatos": "transformador_data_metadatos_tecnicos.txt",
        "variables": "transformador_data_variables.csv",
    }

    # ---- 1) Silver (Delta) ----
    df_principal = None
    if ruta_silver.exists():
        try:
            print(f"   Leyendo Silver (Delta): {ruta_silver}")
            dt = DeltaTable(str(ruta_silver))
            df_principal = dt.to_pandas()
        except Exception as e:
            print(f"   No se pudo leer Delta Silver: {e}")

    # ---- 2) Fallback processed/parquet ----
    if df_principal is None:
        print("     Fallback a archivos en processed/")
        ruta_pq = ruta_processed / archivos_esperados["principal"]
        if not ruta_pq.exists():
            raise FileNotFoundError("No se encontró Silver Delta ni el parquet principal en processed/")
        df_principal = pd.read_parquet(ruta_pq)

    # ---- 3) Índice temporal ----
    if "timestamp" in df_principal.columns:
        df_principal["timestamp"] = pd.to_datetime(df_principal["timestamp"], utc=True, errors="coerce")
        df_principal = df_principal.sort_values("timestamp").set_index("timestamp")
    elif not isinstance(df_principal.index, pd.DatetimeIndex):
        df_principal.index = pd.to_datetime(df_principal.index, utc=True, errors="coerce")
        df_principal = df_principal.sort_index()

    print(f"    Dimensiones: {len(df_principal):,} × {df_principal.shape[1]}")
    if len(df_principal):
        print(f"    Período: {df_principal.index.min()} → {df_principal.index.max()}")
        print(f"    Duración: {df_principal.index.max() - df_principal.index.min()}")

    # ---- 4) Anomalías (parquet o derivado) ----
    df_anomalias = None
    ruta_anom = ruta_processed / archivos_esperados["anomalias"]
    if ruta_anom.exists():
        try:
            print("    Cargando dataset de anomalías (processed/parquet)...")
            df_anomalias = pd.read_parquet(ruta_anom)
            if "timestamp" in df_anomalias.columns:
                df_anomalias["timestamp"] = pd.to_datetime(df_anomalias["timestamp"], utc=True, errors="coerce")
                df_anomalias = df_anomalias.sort_values("timestamp").set_index("timestamp")
            elif not isinstance(df_anomalias.index, pd.DatetimeIndex):
                df_anomalias.index = pd.to_datetime(df_anomalias.index, utc=True, errors="coerce")
                df_anomalias = df_anomalias.sort_index()
        except Exception as e:
            print(f"    Error leyendo anomalías parquet: {e}")

    if df_anomalias is None and "estado_operacional" in df_principal.columns:
        print("    Derivando anomalías desde df_principal (ALERTA/CRITICO)...")
        df_anomalias = df_principal[df_principal["estado_operacional"].isin(["ALERTA", "CRITICO"])].copy()

    # ---- 5) Metadatos (TXT) ----
    metadatos = None
    ruta_meta = ruta_processed / archivos_esperados["metadatos"]
    metadatos_ok = False
    if ruta_meta.exists():
        try:
            metadatos = ruta_meta.read_text(encoding="utf-8")
            metadatos_ok = True
        except Exception as e:
            print(f"   Error cargando metadatos: {e}")

    # ---- 6) Info variables (CSV) ----
    info_variables = None
    categorias_contadas = {}
    ruta_vars = ruta_processed / archivos_esperados["variables"]
    if ruta_vars.exists():
        try:
            info_variables = pd.read_csv(ruta_vars)
            # Conteo por categoría técnica (si existe la columna)
            if "categoria_tecnica" in info_variables.columns:
                categorias_contadas = (
                    info_variables.groupby("categoria_tecnica")["variable"]
                    .count()
                    .sort_index()
                    .to_dict()
                )
        except Exception as e:
            print(f"   Error cargando info de variables: {e}")

    # ---- 7) Distribución estados ----
    # (si existe 'estado_operacional', lo mostramos bonito)
    dist_estados = {}
    if "estado_operacional" in df_principal.columns:
        vc = df_principal["estado_operacional"].value_counts(dropna=False)
        total = len(df_principal) if len(df_principal) else 1
        dist_estados = {k: (int(v), float(v * 100.0 / total)) for k, v in vc.items()}

    # ===== Pretty summary EXACTO como lo quieres =====
    print("\n Distribución de estados operacionales:")
    if dist_estados:
        for estado in ["CRITICO", "ALERTA", "NORMAL"]:  # orden deseado
            if estado in dist_estados:
                n, pct = dist_estados[estado]
                print(f"   {ESTADO_EMOJI.get(estado,'')} {estado}: {n:,} registros ({pct:.1f}%)")
    else:
        print("    (No disponible)")

    if metadatos_ok:
        print("\n📋 Metadatos técnicos cargados exitosamente")
    else:
        print("\n📋 Metadatos técnicos no disponibles")

    print("\n Información de variables:")
    if categorias_contadas:
        # Imprimir en el formato deseado
        # Orden opcional de categorías más comunes en tu dominio
        order = ["Clasificación", "Eléctrica", "Mecánica", "Otra", "Térmica"]
        for cat in order:
            if cat in categorias_contadas:
                print(f"    {cat}: {categorias_contadas[cat]} variables")
        # Imprime cualquier categoría extra que no esté en el orden predefinido
        for cat, cnt in categorias_contadas.items():
            if cat not in order:
                print(f"    {cat}: {cnt} variables")
    else:
        print("    (No disponible)")

    print("\nCARGA COMPLETADA EXITOSAMENTE")
    print("    Datos listos para ingeniería de características")

    return {
        "df_principal": df_principal,
        "df_anomalias": df_anomalias,
        "metadatos": metadatos,
        "info_variables": info_variables,
    }

# Ejecutar
datos_cargados = cargar_datos_transformador_preprocesados(RUTA_SILVER, RUTA_PROCESSED)
df_transformador = datos_cargados["df_principal"]
df_anomalias = datos_cargados["df_anomalias"]
metadatos_tecnicos = datos_cargados["metadatos"]
info_variables = datos_cargados["info_variables"]


⚡ CARGA DE DATOS PREPROCESADOS DE TRANSFORMADORES (Delta First)
   Leyendo Silver (Delta): C:\Users\Asus TUF\Desktop\proy_ml\data\capa_silver\preprocesamiento_silver
    Dimensiones: 8,442 × 15
    Período: 2024-09-10 04:00:00+00:00 → 2025-08-27 21:00:00+00:00
    Duración: 351 days 17:00:00
    Derivando anomalías desde df_principal (ALERTA/CRITICO)...

 Distribución de estados operacionales:
   🚨 CRITICO: 8,429 registros (99.8%)
   ✅ NORMAL: 13 registros (0.2%)

📋 Metadatos técnicos cargados exitosamente

 Información de variables:
    Clasificación: 3 variables
    Eléctrica: 3 variables
    Mecánica: 1 variables
    Otra: 1 variables
    Térmica: 5 variables

CARGA COMPLETADA EXITOSAMENTE
    Datos listos para ingeniería de características


In [8]:
# ================================
# Section 3 — Technical validation (keep original ranges)
# ================================
import pandas as pd

def validar_coherencia_tecnica_transformador(df: pd.DataFrame, parametros: dict) -> dict:
    """
    Realiza validaciones técnicas específicas para datos de transformadores eléctricos.
    Mantiene los mismos rangos fijos definidos en el código original.
    """
    print(" VALIDACIÓN TÉCNICA DE COHERENCIA FÍSICA")
    print("=" * 45)

    validaciones = {}
    alertas = []

    # 1. Validación de coherencia térmica
    if 'temp_spot_hot_value' in df.columns and 'temp_oil_value' in df.columns:
        temp_spot_hot = df['temp_spot_hot_value'].dropna()
        temp_oil = df['temp_oil_value'].dropna()

        indices_comunes = temp_spot_hot.index.intersection(temp_oil.index)
        if len(indices_comunes) > 0:
            coherencia_termica = (temp_spot_hot.loc[indices_comunes] >= temp_oil.loc[indices_comunes]).mean()
            validaciones['coherencia_termica'] = coherencia_termica * 100

            print(f" Coherencia térmica: {coherencia_termica*100:.1f}%")
            if coherencia_termica < 0.95:
                alertas.append(f" Coherencia térmica baja: {coherencia_termica*100:.1f}%")

            gradientes = temp_spot_hot.loc[indices_comunes] - temp_oil.loc[indices_comunes]
            gradiente_medio = gradientes.mean()
            gradiente_max = gradientes.max()

            validaciones['gradiente_termico_medio'] = gradiente_medio
            validaciones['gradiente_termico_max'] = gradiente_max

            print(f"    Gradiente térmico medio: {gradiente_medio:.2f}°C")
            print(f"    Gradiente térmico máximo: {gradiente_max:.2f}°C")

            if gradiente_max > parametros['umbral_gradiente_termico']:
                alertas.append(f" Gradiente térmico excesivo detectado: {gradiente_max:.1f}°C")

    # 2. Validación de consistencia eléctrica
    if 'current_load_value' in df.columns and 'power_apparent_value' in df.columns:
        current = df['current_load_value'].dropna()
        power = df['power_apparent_value'].dropna()

        if len(current) > 0 and len(power) > 0:
            voltaje_estimado = parametros['voltaje_nominal_kv']
            indices_comunes = current.index.intersection(power.index)
            if len(indices_comunes) > 0:
                current_common = current.loc[indices_comunes]
                power_common = power.loc[indices_comunes]

                mask_valido = (current_common > 10) & (power_common > 1)
                if mask_valido.sum() > 0:
                    ratio_potencia = power_common[mask_valido] / current_common[mask_valido]
                    ratio_medio = ratio_potencia.mean()
                    ratio_std = ratio_potencia.std()

                    validaciones['ratio_potencia_corriente'] = ratio_medio
                    validaciones['variabilidad_ratio'] = ratio_std / ratio_medio if ratio_medio > 0 else 0

                    print(f" Ratio P/I medio: {ratio_medio:.3f} kVA/A")
                    print(f"    Coeficiente de variación: {(ratio_std/ratio_medio*100):.1f}%")

    # 3. Validación de rangos operacionales (no cambiar)
    rangos_validos = {
        'temp_oil_value': (0, 120),        # °C
        'temp_spot_hot_value': (0, 150),   # °C
        'temp_ambient_value': (-40, 60),   # °C
        'current_load_value': (0, 5000),   # A
        'power_apparent_value': (0, 200),  # MVA
        'voltage_value': (100, 150),       # kV
        'tap_position_value': (0, 20)      # posición tap
    }

    print(f"\n Validación de rangos operacionales:")
    for variable, (min_val, max_val) in rangos_validos.items():
        if variable in df.columns:
            serie = df[variable].dropna()
            if len(serie) > 0:
                valores_fuera_rango = ((serie < min_val) | (serie > max_val)).sum()
                porcentaje_valido = ((len(serie) - valores_fuera_rango) / len(serie)) * 100

                validaciones[f'validez_{variable}'] = porcentaje_valido

                emoji = '✅' if porcentaje_valido > 95 else '⚠️' if porcentaje_valido > 90 else '❌'
                print(f"   {emoji} {variable}: {porcentaje_valido:.1f}% válido")

                if porcentaje_valido < 90:
                    alertas.append(f" {variable}: {100-porcentaje_valido:.1f}% valores fuera de rango")

    # 4. Validación de continuidad temporal (1H fijo)
    if isinstance(df.index, pd.DatetimeIndex):
        diff_temporal = df.index.to_series().diff()
        freq_esperada = pd.Timedelta(hours=1)

        gaps = diff_temporal[diff_temporal > freq_esperada * 1.5]
        num_gaps = len(gaps)

        validaciones['continuidad_temporal'] = (len(df) - num_gaps) / len(df) * 100
        validaciones['numero_gaps'] = num_gaps

        print(f"\n Continuidad temporal:")
        print(f"    Gaps temporales detectados: {num_gaps}")
        print(f"    Continuidad: {validaciones['continuidad_temporal']:.1f}%")

        if num_gaps > len(df) * 0.05:
            alertas.append(f" Muchos gaps temporales: {num_gaps} ({num_gaps/len(df)*100:.1f}%)")

    # Resumen
    print(f"\n RESUMEN DE VALIDACIÓN TÉCNICA:")
    if not alertas:
        print(f"    Todos los criterios de validación técnica cumplidos")
        validaciones['estado_validacion'] = 'EXITOSO'
    else:
        print(f"    Se detectaron {len(alertas)} alertas de validación:")
        for alerta in alertas:
            print(f"      {alerta}")
        validaciones['estado_validacion'] = 'ADVERTENCIAS'

    validaciones['alertas'] = alertas
    return validaciones

# Ejecutar validación técnica
resultados_validacion = validar_coherencia_tecnica_transformador(df_transformador, PARAMETROS_TRANSFORMADOR)


 VALIDACIÓN TÉCNICA DE COHERENCIA FÍSICA
 Coherencia térmica: 99.9%
    Gradiente térmico medio: 4.67°C
    Gradiente térmico máximo: 14.87°C
 Ratio P/I medio: 0.040 kVA/A
    Coeficiente de variación: 2.5%

 Validación de rangos operacionales:
   ✅ temp_oil_value: 100.0% válido
   ✅ temp_spot_hot_value: 100.0% válido
   ✅ temp_ambient_value: 100.0% válido
   ✅ current_load_value: 100.0% válido
   ✅ power_apparent_value: 100.0% válido
   ✅ voltage_value: 100.0% válido
   ✅ tap_position_value: 100.0% válido

 Continuidad temporal:
    Gaps temporales detectados: 0
    Continuidad: 100.0%

 RESUMEN DE VALIDACIÓN TÉCNICA:
    Todos los criterios de validación técnica cumplidos


In [9]:
# ================================
# Section 4 — Thermal feature engineering (unchanged ranges/logic)
# ================================
import pandas as pd

def crear_features_termicos_avanzados(df: pd.DataFrame, parametros: dict):
    """
    Create advanced thermal features for power transformers.
    Keeps original logic and thresholds as provided.
    """
    print(" CREACIÓN DE FEATURES TÉRMICOS AVANZADOS")
    print("=" * 45)

    df_thermal = df.copy()
    features_creadas = []

    # 1) Fundamental thermal gradients
    print("\n Calculando gradientes térmicos fundamentales...")
    if 'temp_spot_hot_value' in df.columns and 'temp_oil_value' in df.columns:
        # Main thermal gradient (hotspot - oil)
        df_thermal['gradient_hot_oil'] = df['temp_spot_hot_value'] - df['temp_oil_value']
        features_creadas.append('gradient_hot_oil')

        # Normalized gradient (relative to ambient)
        if 'temp_ambient_value' in df.columns:
            df_thermal['gradient_normalized'] = (
                (df['temp_spot_hot_value'] - df['temp_oil_value']) /
                (df['temp_oil_value'] - df['temp_ambient_value'] + 1e-6)  # avoid div by zero
            )
            df_thermal['temp_rise_hot'] = df['temp_spot_hot_value'] - df['temp_ambient_value']
            df_thermal['temp_rise_oil'] = df['temp_oil_value'] - df['temp_ambient_value']
            features_creadas.extend(['gradient_normalized', 'temp_rise_hot', 'temp_rise_oil'])

        print(f"    Gradientes térmicos: {len([f for f in features_creadas if 'gradient' in f or 'rise' in f])} features")

    # 2) Thermal inertia (rates & accelerations)
    print("\n Analizando inercia térmica y velocidades de cambio...")
    variables_termicas = [
        'temp_oil_value', 'temp_spot_hot_value', 'temp_ambient_value',
        'temp_oil_oltc_value', 'temp_bubbling_value'
    ]
    for var in variables_termicas:
        if var in df.columns:
            df_thermal[f'{var}_rate'] = df[var].diff() / 1  # per hour
            df_thermal[f'{var}_accel'] = df_thermal[f'{var}_rate'].diff() / 1
            df_thermal[f'{var}_rate_smooth'] = df_thermal[f'{var}_rate'].rolling(window=6, min_periods=1).mean()
            features_creadas.extend([f'{var}_rate', f'{var}_accel', f'{var}_rate_smooth'])

    print(f"    Features de inercia térmica: {len([f for f in features_creadas if 'rate' in f or 'accel' in f])} features")

    # 3) Thermal efficiency & load relations
    print("\n Calculando eficiencia térmica y relaciones con carga eléctrica...")
    if all(v in df.columns for v in ['temp_oil_value', 'current_load_value', 'temp_ambient_value']):
        df_thermal['thermal_efficiency'] = (
            (df['temp_oil_value'] - df['temp_ambient_value']) /
            (df['current_load_value'] + 1e-6)
        )
        temp_nominal = parametros['temp_aceite_nominal']
        temp_amb_nom = parametros['temp_ambiente_nominal']
        df_thermal['thermal_loading_factor'] = (
            (df['temp_oil_value'] - df['temp_ambient_value']) /
            (temp_nominal - temp_amb_nom)
        )
        features_creadas.extend(['thermal_efficiency', 'thermal_loading_factor'])
        print("    Features de eficiencia térmica: 2 features")

    # 4) Thermal cycles (peaks & valleys)
    print("\n Detectando ciclos térmicos y patrones de fatiga...")
    if 'temp_oil_value' in df.columns:
        try:
            from scipy.signal import find_peaks  # optional dependency
            temp_oil_clean = df['temp_oil_value'].dropna()
            if len(temp_oil_clean) > 48:  # at least 48 hourly points
                peaks_high, _ = find_peaks(
                    temp_oil_clean.values,
                    height=temp_oil_clean.mean(),
                    distance=12  # min 12h between peaks
                )
                peaks_low, _ = find_peaks(
                    -temp_oil_clean.values,
                    height=-temp_oil_clean.mean(),
                    distance=12
                )

                ventana_ciclos = 24 * 7  # 1 week
                df_thermal['thermal_cycles_7d'] = 0.0

                # Map index positions to sequential integer positions
                # to stay consistent with iloc in sliding window
                n_rows = len(df_thermal)
                for i in range(n_rows):
                    inicio = max(0, i - ventana_ciclos)
                    fin = i + 1
                    picos_ventana = len([p for p in peaks_high if inicio <= p < fin])
                    df_thermal.iloc[i, df_thermal.columns.get_loc('thermal_cycles_7d')] = float(picos_ventana)

                features_creadas.append('thermal_cycles_7d')
                print("    Análisis de ciclos térmicos: 1 feature")
        except Exception as e:
            print(f"    Saltando análisis de ciclos (scipy no disponible o error): {e}")

    # 5) Thermal stress indicators (combined)
    print("\n Creando indicadores de estrés térmico combinado...")
    if all(v in df.columns for v in ['temp_spot_hot_value', 'temp_oil_value']):
        temp_max_nominal = 100  # °C (mantener como en el código fuente)
        # Ensure gradient exists (created in step 1)
        if 'gradient_hot_oil' not in df_thermal.columns:
            df_thermal['gradient_hot_oil'] = df['temp_spot_hot_value'] - df['temp_oil_value']
            features_creadas.append('gradient_hot_oil')

        df_thermal['thermal_stress_index'] = (
            0.7 * (df['temp_spot_hot_value'] / temp_max_nominal) +
            0.3 * (df_thermal['gradient_hot_oil'] / parametros['umbral_gradiente_termico'])
        )

        ventana_tendencia = 24 * 30  # 30 días
        df_thermal['overheating_trend'] = (
            df['temp_oil_value'].rolling(window=ventana_tendencia, min_periods=168).mean() -
            df['temp_oil_value'].rolling(window=24, min_periods=12).mean()
        )

        features_creadas.extend(['thermal_stress_index', 'overheating_trend'])
        print("    Indicadores de estrés térmico: 2 features")

    # 6) Gas-related features (bubbling temperature)
    print("\n Analizando temperatura de burbujeo como indicador de gases...")
    if 'temp_bubbling_value' in df.columns:
        bubbling_mean = df['temp_bubbling_value'].rolling(window=24*7, min_periods=24).mean()
        bubbling_std  = df['temp_bubbling_value'].rolling(window=24*7, min_periods=24).std()

        df_thermal['bubbling_anomaly'] = (
            (df['temp_bubbling_value'] - bubbling_mean) / (bubbling_std + 1e-6)
        )
        df_thermal['gas_formation_trend'] = (
            df['temp_bubbling_value'].rolling(window=24*30, min_periods=168).mean() -
            df['temp_bubbling_value'].rolling(window=24*7, min_periods=24).mean()
        )

        features_creadas.extend(['bubbling_anomaly', 'gas_formation_trend'])
        print("    Features de análisis de gases: 2 features")

    # Summary
    total_features_termicas = len(features_creadas)
    print("\n RESUMEN DE FEATURES TÉRMICOS:")
    print(f"   Total features térmicos creados: {total_features_termicas}")
    print(f"    Dimensiones del dataset: {df_thermal.shape[0]:,} × {df_thermal.shape[1]}")

    # Quick quality check on the last N features (unchanged threshold: 50%)
    features_validos = 0
    muestra = features_creadas[-10:] if len(features_creadas) >= 10 else features_creadas
    for feature in muestra:
        if feature in df_thermal.columns:
            valores_validos = df_thermal[feature].notna().sum()
            porcentaje_valido = (valores_validos / len(df_thermal)) * 100 if len(df_thermal) else 0.0
            if porcentaje_valido > 50:
                features_validos += 1

    print(f"    Features con >50% datos válidos: {features_validos}/{len(muestra)} (muestra)")

    return df_thermal, features_creadas

# Ejecutar creación de features térmicos
df_with_thermal, features_termicos = crear_features_termicos_avanzados(df_transformador, PARAMETROS_TRANSFORMADOR)


 CREACIÓN DE FEATURES TÉRMICOS AVANZADOS

 Calculando gradientes térmicos fundamentales...
    Gradientes térmicos: 4 features

 Analizando inercia térmica y velocidades de cambio...
    Features de inercia térmica: 15 features

 Calculando eficiencia térmica y relaciones con carga eléctrica...
    Features de eficiencia térmica: 2 features

 Detectando ciclos térmicos y patrones de fatiga...
    Análisis de ciclos térmicos: 1 feature

 Creando indicadores de estrés térmico combinado...
    Indicadores de estrés térmico: 2 features

 Analizando temperatura de burbujeo como indicador de gases...
    Features de análisis de gases: 2 features

 RESUMEN DE FEATURES TÉRMICOS:
   Total features térmicos creados: 26
    Dimensiones del dataset: 8,442 × 41
    Features con >50% datos válidos: 10/10 (muestra)


In [10]:
# ================================
# Section 5 — Electrical feature engineering 
# ================================
import numpy as np
import pandas as pd

def crear_features_electricos_avanzados(df: pd.DataFrame, parametros: dict):
    """
    Crea features eléctricos especializados para análisis de transformadores.
    Mantiene la lógica y umbrales originales del código fuente.
    """
    print(" CREACIÓN DE FEATURES ELÉCTRICOS AVANZADOS")
    print("=" * 45)

    df_electrical = df.copy()
    features_creadas = []

    # 1) Factores de carga y utilización
    print(" Calculando factores de carga y utilización...")
    if 'current_load_value' in df.columns and 'power_apparent_value' in df.columns:
        capacidad_nominal = parametros['capacidad_nominal_mva'] * 1000.0  # MVA → kVA
        # corriente nominal estimada (A) para trifásico
        corriente_nominal_estimada = capacidad_nominal / (parametros['voltaje_nominal_kv'] * np.sqrt(3))

        df_electrical['load_factor_current'] = df['current_load_value'] / (corriente_nominal_estimada + 1e-6)
        df_electrical['load_factor_power']   = df['power_apparent_value'] / (capacidad_nominal + 1e-6)

        df_electrical['overload_indicator'] = (
            (df_electrical['load_factor_current'] > parametros['umbral_sobrecarga']) |
            (df_electrical['load_factor_power']   > parametros['umbral_sobrecarga'])
        ).astype(int)

        features_creadas.extend(['load_factor_current', 'load_factor_power', 'overload_indicator'])
        print(" Factores de carga: 3 features")

    # 2) Eficiencia y pérdidas eléctricas
    print(" Analizando eficiencia y pérdidas eléctricas...")
    if all(v in df.columns for v in ['current_load_value', 'voltage_value', 'power_apparent_value']):
        # S_teórica (kVA) ≈ √3 * V(kV) * I(A) / 1000
        potencia_teorica = (np.sqrt(3) * df['voltage_value'] * df['current_load_value']) / 1000.0
        df_electrical['efficiency_indicator'] = df['power_apparent_value'] / (potencia_teorica + 1e-6)

        # FP estimado (P ≈ 0.9·S → FP ≈ 0.9)
        factor_potencia_estimado = 0.9
        potencia_activa_estimada = df['power_apparent_value'] * factor_potencia_estimado
        df_electrical['power_factor_estimated'] = potencia_activa_estimada / (df['power_apparent_value'] + 1e-6)

        df_electrical['relative_losses'] = 1.0 - df_electrical['efficiency_indicator']

        features_creadas.extend(['efficiency_indicator', 'power_factor_estimated', 'relative_losses'])
        print(" Features de eficiencia: 3 features")

    # 3) Sistema OLTC
    print(" Analizando comportamiento del sistema OLTC...")
    if 'tap_position_value' in df.columns:
        tap_changes = df['tap_position_value'].diff().abs()

        for ventana in [24, 168, 720]:  # 1 día, 1 semana, 1 mes
            nombre_dias = ventana // 24
            df_electrical[f'tap_operations_{nombre_dias}d'] = tap_changes.rolling(window=ventana, min_periods=1).sum()
            features_creadas.append(f'tap_operations_{nombre_dias}d')

        tap_central = 8.5
        df_electrical['tap_deviation_center'] = (df['tap_position_value'] - tap_central).abs()
        df_electrical['tap_extreme_position'] = (
            (df['tap_position_value'] <= 2) | (df['tap_position_value'] >= 15)
        ).astype(int)
        df_electrical['tap_instability'] = tap_changes.rolling(window=24, min_periods=1).std()

        features_creadas.extend(['tap_deviation_center', 'tap_extreme_position', 'tap_instability'])
        print(f" Features OLTC: {len([f for f in features_creadas if 'tap' in f])} features")

    # 4) Indicadores de estabilidad eléctrica
    print(" Creando indicadores de estabilidad eléctrica...")
    variables_electricas = ['current_load_value', 'voltage_value', 'power_apparent_value']
    for var in variables_electricas:
        if var in df.columns:
            cv_ventana = 24
            media_rolling = df[var].rolling(window=cv_ventana, min_periods=6).mean()
            std_rolling   = df[var].rolling(window=cv_ventana, min_periods=6).std()
            df_electrical[f'{var}_stability'] = std_rolling / (media_rolling + 1e-6)

            df_electrical[f'{var}_transient'] = (
                (df[var].diff().abs()) > (df[var].rolling(window=168).std() * 2)
            ).astype(int)

            features_creadas.extend([f'{var}_stability', f'{var}_transient'])

    print(f" Indicadores de estabilidad: {len([f for f in features_creadas if 'stability' in f or 'transient' in f])} features")

    # 5) Combinaciones eléctricas avanzadas
    print(" Creando features de combinaciones eléctricas...")
    if all(v in df.columns for v in ['current_load_value', 'voltage_value']) and \
       all(f in df_electrical.columns for f in ['load_factor_current', 'efficiency_indicator']):
        df_electrical['electrical_stress_index'] = (
            0.6 * df_electrical['load_factor_current'] +
            0.4 * (1.0 - df_electrical['efficiency_indicator'])
        )
        df_electrical['suboptimal_operation'] = (
            (df_electrical['load_factor_current'] < 0.3) |
            (df_electrical['load_factor_current'] > 1.1) |
            (df_electrical['efficiency_indicator'] < 0.95)
        ).astype(int)

        features_creadas.extend(['electrical_stress_index', 'suboptimal_operation'])
        print(" Combinaciones eléctricas: 2 features")

    # 6) Tendencias de degradación
    print(" Analizando tendencias de degradación eléctrica...")
    if 'efficiency_indicator' in df_electrical.columns:
        ventana_tendencia = 24 * 30  # 30 días
        df_electrical['efficiency_degradation_trend'] = (
            df_electrical['efficiency_indicator'].rolling(window=ventana_tendencia, min_periods=168).mean() -
            df_electrical['efficiency_indicator'].rolling(window=24, min_periods=12).mean()
        )
        df_electrical['efficiency_degradation_accel'] = df_electrical['efficiency_degradation_trend'].diff()

        features_creadas.extend(['efficiency_degradation_trend', 'efficiency_degradation_accel'])
        print(" Tendencias de degradación: 2 features")

    # Resumen
    total_features_electricas = len(features_creadas)
    print(" RESUMEN DE FEATURES ELÉCTRICOS:")
    print(f" Total features eléctricos creados: {total_features_electricas}")
    print(f" Dimensiones del dataset: {df_electrical.shape[0]:,} × {df_electrical.shape[1]}")

    # Chequeo rápido de completitud (misma lógica del original)
    muestra = features_creadas[-10:] if len(features_creadas) >= 10 else features_creadas
    features_validos = 0
    for f in muestra:
        if f in df_electrical.columns:
            pct_ok = (df_electrical[f].notna().sum() / len(df_electrical) * 100.0) if len(df_electrical) else 0.0
            if pct_ok > 50:
                features_validos += 1
    print(f" Features con >50% datos válidos: {features_validos}/{len(muestra)} (muestra)")

    return df_electrical, features_creadas

# Ejecutar creación de features eléctricos
df_with_electrical, features_electricos = crear_features_electricos_avanzados(df_with_thermal, PARAMETROS_TRANSFORMADOR)


 CREACIÓN DE FEATURES ELÉCTRICOS AVANZADOS
 Calculando factores de carga y utilización...
 Factores de carga: 3 features
 Analizando eficiencia y pérdidas eléctricas...
 Features de eficiencia: 3 features
 Analizando comportamiento del sistema OLTC...
 Features OLTC: 6 features
 Creando indicadores de estabilidad eléctrica...
 Indicadores de estabilidad: 7 features
 Creando features de combinaciones eléctricas...
 Combinaciones eléctricas: 2 features
 Analizando tendencias de degradación eléctrica...
 Tendencias de degradación: 2 features
 RESUMEN DE FEATURES ELÉCTRICOS:
 Total features eléctricos creados: 22
 Dimensiones del dataset: 8,442 × 63
 Features con >50% datos válidos: 10/10 (muestra)


In [ ]:
# ================================
# Section 6 — Labeling (binaria, multiclase, RUL, severidad)
# ================================
from duckdb import df
import numpy as np
import pandas as pd

def crear_etiquetas_prediccion_transformador(df_principal: pd.DataFrame,
                                             df_anomalias: pd.DataFrame,
                                             parametros: dict):
    """
    Crea etiquetas de predicción para mantenimiento predictivo de transformadores.
    Mantiene la lógica y umbrales originales del código fuente.
    """
    print("    CREACIÓN DE ETIQUETAS DE PREDICCIÓN")
    print("=" * 40)

    df_labeled = df_principal.copy()
    horizonte_dias  = parametros['horizonte_prediccion_dias']
    horizonte_horas = horizonte_dias * 24

    print(f"    Horizonte de predicción: {horizonte_dias} días ({horizonte_horas} horas)")

    # Asegurar índice temporal
    if "timestamp" in df_labeled.columns:
        df_labeled["timestamp"] = pd.to_datetime(df_labeled["timestamp"], utc=True, errors="coerce")
        df_labeled = df_labeled.sort_values("timestamp").set_index("timestamp")
    elif not isinstance(df_labeled.index, pd.DatetimeIndex):
        df_labeled.index = pd.to_datetime(df_labeled.index, utc=True, errors="coerce")
        df_labeled = df_labeled.sort_index()

    # Inicializar etiquetas
    df_labeled['falla_30d'] = 0
    df_labeled['estado_futuro'] = 'NORMAL'
    df_labeled['rul_dias'] = float(horizonte_dias)
    df_labeled['severidad_futura'] = 0.0

    # -----------------------------
    # 1) Identificar eventos críticos
    # -----------------------------
    print("    Identificando eventos críticos...")
    eventos_criticos = []

    if df_anomalias is not None and len(df_anomalias) > 0:
        df_anom = df_anomalias.copy()
        if "timestamp" in df_anom.columns:
            df_anom["timestamp"] = pd.to_datetime(df_anom["timestamp"], utc=True, errors="coerce")
            df_anom = df_anom.sort_values("timestamp").set_index("timestamp")
        elif not isinstance(df_anom.index, pd.DatetimeIndex):
            df_anom.index = pd.to_datetime(df_anom.index, utc=True, errors="coerce")
            df_anom = df_anom.sort_index()

        if 'estado_operacional' in df_anom.columns:
            eventos_criticos = df_anom[df_anom['estado_operacional'] == 'CRITICO'].index.tolist()
        elif 'nivel_severidad' in df_anom.columns:
            eventos_criticos = df_anom[df_anom['nivel_severidad'] >= 2].index.tolist()

    if not eventos_criticos:
        print(" No se encontraron marcadores de eventos críticos")
    else:
        print(f"    Eventos críticos identificados: {len(eventos_criticos)}")
        print(f"    Período de eventos: {min(eventos_criticos)} a {max(eventos_criticos)}")

    # -----------------------------
    # 2) Etiqueta binaria + RUL
    # -----------------------------
    print("    Creando etiquetas binarias de predicción...")
    eventos_etiquetados = 0
    registros_positivos = 0

    for evento_critico in eventos_criticos:
        inicio_ventana = evento_critico - pd.Timedelta(hours=horizonte_horas)
        fin_ventana    = evento_critico

        mask_ventana = (df_labeled.index >= inicio_ventana) & (df_labeled.index < fin_ventana)
        idxs = df_labeled.index[mask_ventana]

        if len(idxs) > 0:
            df_labeled.loc[idxs, 'falla_30d'] = 1
            # RUL por timestamp
            tdiff_horas = (evento_critico - idxs).total_seconds() / 3600.0
            df_labeled.loc[idxs, 'rul_dias'] = np.maximum(0.0, tdiff_horas / 24.0)

            eventos_etiquetados += 1
            registros_positivos += len(idxs)

    print(f"    Eventos procesados: {eventos_etiquetados}")
    print(f"    Registros marcados como positivos: {registros_positivos:,}")

    # -----------------------------
    # 3) Etiquetas multiclase por RUL
    # -----------------------------
    print("    Creando etiquetas multi-clase...")
    umbral_critico = 7   # ≤7 días
    umbral_alerta  = 15  # 7–15 días

    mask_critico = df_labeled['rul_dias'] <= umbral_critico
    mask_alerta  = (df_labeled['rul_dias'] > umbral_critico) & (df_labeled['rul_dias'] <= umbral_alerta)

    df_labeled.loc[mask_critico, 'estado_futuro'] = 'CRITICO'
    df_labeled.loc[mask_alerta,  'estado_futuro'] = 'ALERTA'

    dist = df_labeled['estado_futuro'].value_counts()
    print("    Distribución de estados futuros:")
    for estado, cantidad in dist.items():
        porcentaje = (cantidad / len(df_labeled) * 100.0) if len(df_labeled) else 0.0
        emoji = {'NORMAL': '✅', 'ALERTA': '⚠️', 'CRITICO': '🚨'}.get(estado, '📊')
        print(f"      {emoji} {estado}: {cantidad:,} ({porcentaje:.1f}%)")

    # -----------------------------
    # 4) Severidad progresiva (0–100%)
    # -----------------------------
    print("    Creando etiquetas de severidad progresiva...")
    df_labeled['severidad_futura'] = 100.0 * (1.0 - df_labeled['rul_dias'] / float(horizonte_dias))
    df_labeled['severidad_futura'] = df_labeled['severidad_futura'].clip(0, 100)

    # -----------------------------
    # 5) Extras de etiquetado
    # -----------------------------
    print("   Creando features adicionales de etiquetado...")
    df_labeled['dias_proximo_evento'] = float(horizonte_dias)

    # días hasta próximo evento (lineal)
    if eventos_criticos:
        eventos_sorted = sorted(eventos_criticos)
        # vectorizado por búsqueda binaria sobre índices si quieres optimizar; aquí mantenemos simple/legible
        for i, ts in enumerate(df_labeled.index):
            ev_futuros = [e for e in eventos_sorted if e > ts]
            if ev_futuros:
                proximo = ev_futuros[0]
                dias = (proximo - ts).total_seconds() / (24 * 3600.0)
                df_labeled.iloc[i, df_labeled.columns.get_loc('dias_proximo_evento')] = min(dias, horizonte_dias)


    df["severidad_futura"] = (100.0 * (1.0 - df["rul_dias"] / float(horizonte_dias))).clip(0, 100)
    df["proximidad_evento"] = np.exp(-df["dias_proximo_evento"] / 10.0)
    df["riesgo_acumulativo"] = df["riesgo_acumulativo"] / df["riesgo_acumulativo"].max()

    # proximidad exponencial
    df_labeled['proximidad_evento'] = np.exp(-df_labeled['dias_proximo_evento'] / 10.0)

    # riesgo acumulativo (decay exponencial respecto a múltiples eventos)
    df_labeled['riesgo_acumulativo'] = 0.0
    for evento in eventos_criticos:
        diff_horas = (np.abs(df_labeled.index - evento).total_seconds()) / 3600.0
        influencia = np.exp(-(diff_horas) / (horizonte_horas / 2.0))
        df_labeled['riesgo_acumulativo'] += influencia

    # normalizar a [0,1]
    max_riesgo = df_labeled['riesgo_acumulativo'].max()
    if pd.notna(max_riesgo) and max_riesgo > 0:
        df_labeled['riesgo_acumulativo'] = df_labeled['riesgo_acumulativo'] / max_riesgo

    # -----------------------------
    # 6) Validación rápida de etiquetas
    # -----------------------------
    print("    Validando calidad de etiquetas...")
    tasa_positivos = float(df_labeled['falla_30d'].mean()) if len(df_labeled) else 0.0
    print(f"    Tasa de casos positivos: {tasa_positivos:.3f} ({tasa_positivos*100:.1f}%)")

    if tasa_positivos < 0.01:
        print("    Advertencia: Muy pocos casos positivos (<1%)")
    elif tasa_positivos > 0.5:
        print("    Advertencia: Demasiados casos positivos (>50%)")
    else:
        print("   Balance de clases apropiado para mantenimiento predictivo")

    rul_valido = ((df_labeled['rul_dias'] >= 0) & (df_labeled['rul_dias'] <= horizonte_dias)).mean() if len(df_labeled) else 1.0
    print(f"   RUL válido: {rul_valido*100:.1f}% de registros")

    severidad_media = float(df_labeled['severidad_futura'].mean()) if len(df_labeled) else 0.0
    severidad_max   = float(df_labeled['severidad_futura'].max()) if len(df_labeled) else 0.0
    print(f"    Severidad media: {severidad_media:.1f}%, máxima: {severidad_max:.1f}%")

    etiquetas_creadas = [
        'falla_30d', 'estado_futuro', 'rul_dias', 'severidad_futura',
        'dias_proximo_evento', 'proximidad_evento', 'riesgo_acumulativo'
    ]

    print("    RESUMEN DE ETIQUETADO:")
    print(f"    Etiquetas creadas: {len(etiquetas_creadas)}")
    print(f"    Dataset etiquetado: {df_labeled.shape[0]:,} × {df_labeled.shape[1]}")
    print("    Listo para entrenamiento de modelos predictivos")

    return df_labeled, etiquetas_creadas

# Ejecutar creación de etiquetas
df_with_labels, etiquetas_creadas = crear_etiquetas_prediccion_transformador(
    df_with_electrical, df_anomalias, PARAMETROS_TRANSFORMADOR
)


    CREACIÓN DE ETIQUETAS DE PREDICCIÓN
    Horizonte de predicción: 30 días (720 horas)
    Identificando eventos críticos...
    Eventos críticos identificados: 8429
    Período de eventos: 2024-09-10 04:00:00+00:00 a 2025-08-27 21:00:00+00:00
    Creando etiquetas binarias de predicción...
    Eventos procesados: 8428
    Registros marcados como positivos: 5,809,320
    Creando etiquetas multi-clase...
    Distribución de estados futuros:
      ✅ NORMAL: 8,082 (95.7%)
      ⚠️ ALERTA: 192 (2.3%)
      🚨 CRITICO: 168 (2.0%)
    Creando etiquetas de severidad progresiva...
   Creando features adicionales de etiquetado...
    Validando calidad de etiquetas...
    Tasa de casos positivos: 1.000 (100.0%)
    Advertencia: Demasiados casos positivos (>50%)
   RUL válido: 100.0% de registros
    Severidad media: 4.3%, máxima: 99.9%
    RESUMEN DE ETIQUETADO:
    Etiquetas creadas: 7
    Dataset etiquetado: 8,442 × 70
    Listo para entrenamiento de modelos predictivos


In [ ]:
# ================================
# Section 7 — Finalize, optimize, split, and save (Parquet/CSV + Delta)
# ================================
from datetime import datetime
import numpy as np
import pandas as pd
from deltalake import write_deltalake

def finalizar_dataset_transformador(
    df_final: pd.DataFrame,
    features_termicos: list,
    features_electricos: list,
    etiquetas_creadas: list,
    ruta_features: Path,               # carpeta para Parquet/CSV/metadata (legado de notebook)
    parametros: dict,
) -> dict:
    """
    Finaliza, optimiza y guarda el dataset completo de features para transformadores.
    - Mantiene tu flujo original (resumen, optimización, split 80/20, Parquet/CSV, metadatos TXT)
    - Agrega guardado en Delta Lake (Gold) particionado por year/month:
        * RUTA_GOLD_COMPLETE, RUTA_GOLD_TRAIN, RUTA_GOLD_VALID (definidos antes)
    """
    print(" FINALIZACIÓN Y GUARDADO DEL DATASET")
    print("=" * 42)

    # 1) Resumen de features
    print(" Análisis completo de features creados...")

    variables_originales = [
        'current_load_value', 'power_apparent_value', 'tap_position_value',
        'temp_oil_value', 'temp_oil_oltc_value', 'temp_ambient_value',
        'temp_bubbling_value', 'temp_spot_hot_value', 'voltage_value',
        'estado_operacional', 'nivel_severidad', 'variables_anomalas', 'descripcion_anomalia'
    ]

    features_por_categoria = {
        'Variables Originales': [c for c in df_final.columns if c in variables_originales],
        'Features Térmicos': features_termicos,
        'Features Eléctricos': features_electricos,
        'Etiquetas de Predicción': etiquetas_creadas,
        'Otros Features': []
    }

    todos_categorizados = []
    for lst in features_por_categoria.values():
        todos_categorizados.extend(lst)

    features_por_categoria['Otros Features'] = [c for c in df_final.columns if c not in todos_categorizados]

    print(f"   Dimensiones del dataset final: {df_final.shape[0]:,} × {df_final.shape[1]}")
    print(f"    Distribución de features por categoría:")
    total_features = 0
    for cat, lst in features_por_categoria.items():
        cnt = len(lst)
        total_features += cnt
        if cnt > 0:
            print(f"       {cat}: {cnt} features")
    print(f"    Total features: {total_features}")

    # 2) Optimización de memoria (misma lógica)
    print("    Optimizando memoria y tipos de datos...")
    memoria_inicial = df_final.memory_usage(deep=True).sum() / 1024**2
    df_optimized = df_final.copy()

    for col in df_optimized.columns:
        if df_optimized[col].dtype == 'float64':
            if df_optimized[col].notna().any():
                mn, mx = df_optimized[col].min(), df_optimized[col].max()
                if np.isfinite(mn) and np.isfinite(mx) and abs(mn) < 1e37 and abs(mx) < 1e37:
                    df_optimized[col] = df_optimized[col].astype('float32')

        elif df_optimized[col].dtype == 'int64':
            if df_optimized[col].notna().any():
                mn, mx = df_optimized[col].min(), df_optimized[col].max()
                if mn >= 0 and mx <= 255:
                    df_optimized[col] = df_optimized[col].astype('uint8')
                elif mn >= -128 and mx <= 127:
                    df_optimized[col] = df_optimized[col].astype('int8')
                elif mn >= 0 and mx <= 65535:
                    df_optimized[col] = df_optimized[col].astype('uint16')
                elif mn >= -32768 and mx <= 32767:
                    df_optimized[col] = df_optimized[col].astype('int16')
                else:
                    df_optimized[col] = df_optimized[col].astype('int32')

    memoria_final = df_optimized.memory_usage(deep=True).sum() / 1024**2
    reduccion_memoria = (memoria_inicial - memoria_final) / max(memoria_inicial, 1e-9) * 100.0

    print(f"   Memoria inicial: {memoria_inicial:.1f} MB")
    print(f"    Memoria optimizada: {memoria_final:.1f} MB")
    print(f"    Reducción de memoria: {reduccion_memoria:.1f}%")

    # 3) Split temporal 80/20 (mismo criterio)
    print("    Realizando división temporal del dataset...")
    # Asegurar índice datetime
    if "timestamp" in df_optimized.columns:
        df_optimized["timestamp"] = pd.to_datetime(df_optimized["timestamp"], utc=True, errors="coerce")
        df_optimized = df_optimized.sort_values("timestamp").set_index("timestamp")
    elif not isinstance(df_optimized.index, pd.DatetimeIndex):
        df_optimized.index = pd.to_datetime(df_optimized.index, utc=True, errors="coerce")
        df_optimized = df_optimized.sort_index()

    total_registros = len(df_optimized)
    indice_division = int(total_registros * 0.8)
    fecha_division = df_optimized.index[indice_division] if total_registros else pd.NaT

    df_train = df_optimized.iloc[:indice_division].copy()
    df_validation = df_optimized.iloc[indice_division:].copy()

    print(f"   Dataset de entrenamiento: {len(df_train):,} registros ({len(df_train)/total_registros*100:.1f}%)")
    print(f"   Dataset de validación: {len(df_validation):,} registros ({len(df_validation)/total_registros*100:.1f}%)")
    print(f"   Fecha de división: {fecha_division}")
    if len(df_train):
        print(f"   Entrenamiento: {df_train.index.min()} → {df_train.index.max()}")
    if len(df_validation):
        print(f"   Validación: {df_validation.index.min()} → {df_validation.index.max()}")

    if 'falla_30d' in df_optimized.columns:
        print(f"   Tasa de fallas en entrenamiento: {df_train['falla_30d'].mean():.3f}")
        print(f"   Tasa de fallas en validación: {df_validation['falla_30d'].mean():.3f}")

    # 4) Guardado en múltiples formatos (Parquet/CSV) + Delta Lake (Gold)
    print("   Guardando datasets (Parquet/CSV + Delta Lake)...")
    archivos_guardados = {}
    timestamp = datetime.now().strftime('%Y%m%d_%H%M')

    # --- Guardado Parquet/CSV (como en tu notebook) ---
    try:
        ruta_features.mkdir(parents=True, exist_ok=True)

        # Completo
        archivo_completo_parquet = ruta_features / f'transformer_features_complete_{timestamp}.parquet'
        archivo_completo_csv     = ruta_features / f'transformer_features_complete_{timestamp}.csv'
        df_optimized.to_parquet(archivo_completo_parquet, compression='snappy')
        df_optimized.to_csv(archivo_completo_csv, index=True)

        archivos_guardados['dataset_completo'] = {
            'parquet': archivo_completo_parquet,
            'csv': archivo_completo_csv,
            'registros': len(df_optimized),
            'features': df_optimized.shape[1]
        }

        # Train
        archivo_train_parquet = ruta_features / f'transformer_features_train_{timestamp}.parquet'
        df_train.to_parquet(archivo_train_parquet, compression='snappy')
        archivos_guardados['dataset_train'] = {
            'parquet': archivo_train_parquet,
            'registros': len(df_train),
            'features': df_train.shape[1]
        }

        # Validation
        archivo_val_parquet = ruta_features / f'transformer_features_validation_{timestamp}.parquet'
        df_validation.to_parquet(archivo_val_parquet, compression='snappy')
        archivos_guardados['dataset_validation'] = {
            'parquet': archivo_val_parquet,
            'registros': len(df_validation),
            'features': df_validation.shape[1]
        }

        print(f"    Dataset completo guardado: {df_optimized.shape}")
        print(f"    Dataset entrenamiento guardado: {df_train.shape}")
        print(f"    Dataset validación guardado: {df_validation.shape}")
    except Exception as e:
        print(f"    Error guardando Parquet/CSV: {e}")
        archivos_guardados['error_parquet_csv'] = str(e)

    # --- Guardado Delta Lake (Gold) ---
    try:
        def _save_delta(df_in: pd.DataFrame, path_dir: Path):
            if df_in.empty:
                return
            out = df_in.reset_index().copy()
            out["year"]  = out["timestamp"].dt.year.astype("int32")
            out["month"] = out["timestamp"].dt.month.astype("int8")
            path_dir.mkdir(parents=True, exist_ok=True)
            write_deltalake(str(path_dir), out, mode="overwrite", partition_by=["year", "month"])

        _save_delta(df_optimized, RUTA_GOLD_COMPLETE)
        _save_delta(df_train,     RUTA_GOLD_TRAIN)
        _save_delta(df_validation,RUTA_GOLD_VALID)

        archivos_guardados['delta'] = {
            'complete': RUTA_GOLD_COMPLETE,
            'train':    RUTA_GOLD_TRAIN,
            'valid':    RUTA_GOLD_VALID
        }
        print("   Tablas Delta (Gold) guardadas (particionado year/month)")
    except Exception as e:
        print(f"   Error guardando Delta Gold: {e}")
        archivos_guardados['error_delta'] = str(e)

    # 5) Documentación técnica (TXT) — mismo contenido, con saltos de línea
    print(" Generando documentación técnica completa...")
    try:
        archivo_metadatos = ruta_features / f'transformer_features_metadata_{timestamp}.txt'
        with open(archivo_metadatos, 'w', encoding='utf-8') as f:
            w = f.write
            w("DOCUMENTACIÓN TÉCNICA - FEATURES DE TRANSFORMADORES ELÉCTRICOS\n")
            w("=" * 70 + "\n\n")

            # Información general
            w(f"Generado: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
            w("Notebook/Base: 03_feature_engineering_transformadores\n")
            w("Objetivo: Mantenimiento Predictivo de Transformadores\n")
            w(f"Horizonte de predicción: {parametros['horizonte_prediccion_dias']} días\n\n")

            # Dimensiones y estadísticas
            w("DIMENSIONES DEL DATASET\n")
            w("-" * 25 + "\n")
            w(f"Total de registros: {df_optimized.shape[0]:,}\n")
            w(f"Total de features: {df_optimized.shape[1]}\n")
            w(f"Período temporal: {df_optimized.index.min()} → {df_optimized.index.max()}\n")
            w(f"Frecuencia: {parametros['frecuencia_muestreo']} (horaria)\n")
            w(f"Memoria utilizada: {memoria_final:.1f} MB\n\n")

            # Features por categoría
            w("FEATURES POR CATEGORÍA TÉCNICA\n")
            w("-" * 32 + "\n")
            for categoria, lista in features_por_categoria.items():
                if len(lista) > 0:
                    w(f"{categoria} ({len(lista)} features):\n")
                    for i, feat in enumerate(lista, 1):
                        w(f"  {i:2d}. {feat}\n")
            w("\n")

            # Etiquetas
            w("ETIQUETAS DE PREDICCIÓN\n")
            w("-" * 22 + "\n")
            w("Etiqueta binaria: falla_30d\n")
            w("Etiqueta multi-clase: estado_futuro\n")
            w("RUL (Remaining Useful Life): rul_dias\n")
            w("Severidad progresiva: severidad_futura\n\n")

            # División temporal
            w("DIVISIÓN TEMPORAL\n")
            w("-" * 16 + "\n")
            w(f"Entrenamiento: {len(df_train):,} registros ({len(df_train)/total_registros*100:.1f}%)\n")
            w(f"Validación: {len(df_validation):,} registros ({len(df_validation)/total_registros*100:.1f}%)\n")
            w(f"Fecha de división: {fecha_division}\n\n")

            # Parámetros técnicos
            w("PARÁMETROS TÉCNICOS\n")
            w("-" * 18 + "\n")
            for k, v in parametros.items():
                w(f"{k}: {v}\n")

        archivos_guardados['metadatos'] = archivo_metadatos
        print("    Documentación técnica generada")
    except Exception as e:
        print(f"    Error generando documentación: {e}")

    # 6) Resumen final
    print("    FEATURE ENGINEERING COMPLETADO EXITOSAMENTE")
    print("=" * 50)
    print(f"    Dataset final: {df_optimized.shape[0]:,} registros × {df_optimized.shape[1]} features")
    print(f"    Features creados: {len(features_termicos + features_electricos)} especializados")
    print(f"    Etiquetas de predicción: {len(etiquetas_creadas)} tipos")
    num_ds = len([k for k, v in archivos_guardados.items() if isinstance(v, dict) and ('parquet' in v or 'complete' in v)])
    print(f"    Archivos guardados: {num_ds} datasets")
    print(f"    Documentación: Completa y detallada")
    print("    DATASET LISTO PARA ENTRENAMIENTO DE MODELOS DE ML")

    if 'falla_30d' in df_optimized.columns:
        tasa_positivos_final = float(df_optimized['falla_30d'].mean())
        print("    ESTADÍSTICAS CLAVE PARA MODELADO:")
        print(f"    Tasa de casos positivos: {tasa_positivos_final:.3f} ({tasa_positivos_final*100:.1f}%)")
        if 'rul_dias' in df_optimized.columns:
            print(f"    RUL medio: {df_optimized['rul_dias'].mean():.1f} días "
                  f"(rango: {df_optimized['rul_dias'].min():.1f} - {df_optimized['rul_dias'].max():.1f})")
        if 'severidad_futura' in df_optimized.columns:
            print(f"   Severidad media: {df_optimized['severidad_futura'].mean():.1f}%")

    return {
        'dataset_final': df_optimized,
        'dataset_train': df_train,
        'dataset_validation': df_validation,
        'archivos_guardados': archivos_guardados,
        'estadisticas': {
            'total_features': total_features,
            'memoria_mb': memoria_final,
            'reduccion_memoria_pct': reduccion_memoria,
            'fecha_division': fecha_division
        }
    }

# Ejecutar finalización del dataset
resultado_final = finalizar_dataset_transformador(
    df_with_labels,
    features_termicos,
    features_electricos,
    etiquetas_creadas,
    RUTA_GOLD_BASE,                # carpeta para Parquet/CSV/TXT (ej. BASE_DIR / "data/features")
    PARAMETROS_TRANSFORMADOR
)


💾 FINALIZACIÓN Y GUARDADO DEL DATASET
📊 Análisis completo de features creados...
   📈 Dimensiones del dataset final: 8,443 × 70
   📊 Distribución de features por categoría:
      🔧 Variables Originales: 13 features
      🔧 Features Térmicos: 26 features
      🔧 Features Eléctricos: 22 features
      🔧 Etiquetas de Predicción: 7 features
      🔧 Otros Features: 2 features
   ✅ Total features: 70
   🚀 Optimizando memoria y tipos de datos...
   📊 Memoria inicial: 7.3 MB
   📊 Memoria optimizada: 5.1 MB
   ✅ Reducción de memoria: 30.7%
   📅 Realizando división temporal del dataset...
   📊 Dataset de entrenamiento: 6,754 registros (80.0%)
   📊 Dataset de validación: 1,689 registros (20.0%)
   📅 Fecha de división: 2025-06-18 14:00:00+00:00
   📅 Entrenamiento: 2024-09-10 04:00:00+00:00 → 2025-06-18 13:00:00+00:00
   📅 Validación: 2025-06-18 14:00:00+00:00 → 2025-08-27 22:00:00+00:00
   📊 Tasa de fallas en entrenamiento: 1.000
   📊 Tasa de fallas en validación: 0.999
   💾 Guardando datasets (Pa